# LoRA Fine-Tuning on Google Colab

**Model:** `Qwen/Qwen3-VL-4B-Instruct` + LoRA adapters (PEFT).

> **The baseline must be complete and saved before running this.**
> Section 2 refuses to continue otherwise — the experiment requires the base
> model to be measured *before* fine-tuning.

| Setting | Value |
|---|---|
| Train / validation | 838 / 184 examples |
| LoRA r / alpha / dropout | 16 / 32 / 0.05 |
| Batch × grad-accum | 1 × 8 (effective 8) |
| Learning rate | 1e-4 |
| Epochs | 3 |
| Precision | fp16 on T4 |
| Quantization | 4-bit NF4 |
| Vision tower | frozen |
| Seed | 42 |

> **Runtime → Change runtime type → T4 GPU** before running.


## 1. Setup


In [ ]:
# ============================================================
#  ONE-CELL SETUP — safe to re-run after any kernel restart
# ============================================================
import os, sys, time, shutil, zipfile, json
from pathlib import Path

import torch
assert torch.cuda.is_available(), 'No CUDA GPU. Runtime -> Change runtime type -> T4 GPU.'
props = torch.cuda.get_device_properties(0)
GPU_NAME, GPU_MEM_GB = props.name, round(props.total_memory / 1e9, 1)
print(f'GPU        : {GPU_NAME} | {GPU_MEM_GB} GB | compute {props.major}.{props.minor}')
print(f'native bf16: {props.major >= 8} (False on T4 -> fp16)')

IN_COLAB = 'google.colab' in sys.modules
PROJECT = Path('/content/casting-defect-vlm')

def _looks_like_project(d):
    return (d / 'config' / 'config.yaml').exists() and (d / 'src' / 'prepare_data.py').exists()

if not _looks_like_project(PROJECT):
    found = [d for d in Path('/content').iterdir() if d.is_dir() and _looks_like_project(d)]
    if found:
        PROJECT = found[0]
    elif IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
        MARKERS = ['config/config.yaml', 'src/prompts.py',
                   'scripts/run_baseline.py', 'results/baseline/eval_subset.json']
        best = None
        for zp in sorted(Path('/content/drive/MyDrive').glob('*.zip')):
            try:
                with zipfile.ZipFile(zp) as zf:
                    names = zf.namelist()
            except Exception:
                continue
            tops = {n.split('/')[0] for n in names if '/' in n}
            pre = (next(iter(tops)) + '/') if len(tops) == 1 else ''
            stripped = {n[len(pre):] if pre and n.startswith(pre) else n for n in names}
            if all(m in stripped for m in MARKERS):
                best = zp; break
        assert best, 'No project ZIP found in MyDrive.'
        print(f'extracting {best.name} ...')
        with zipfile.ZipFile(best) as zf:
            zf.extractall('/content')
        cands = [d for d in Path('/content').iterdir() if d.is_dir() and _looks_like_project(d)]
        assert cands, 'extracted but project folder not found'
        PROJECT = cands[0]
    else:
        raise SystemExit('project not found')
else:
    print('project already extracted — reusing')

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print('PROJECT    :', PROJECT)

# portable image paths (idempotent)
target = PROJECT / 'src' / 'prepare_data.py'
MARKER = '# --- portability hotfix ---'
_src = target.read_text()
if MARKER not in _src:
    target.write_text(_src + '\n\n' + MARKER + '''
def load_jsonl(path, image_root=None, repair_image_paths=True):
    """Load JSONL, rebuilding absolute image paths for THIS machine."""
    import json as _json
    from .data_analysis import find_dataset_root
    from .utils import load_config, resolve_path
    p = resolve_path(path)
    if not p.exists():
        raise FileNotFoundError(f"{p} not found.")
    with p.open("r", encoding="utf-8") as fh:
        examples = [_json.loads(line) for line in fh if line.strip()]
    if not repair_image_paths:
        return examples
    if image_root is None:
        image_root = find_dataset_root(load_config()["data"]["raw_dir"])
    root = resolve_path(image_root)
    for ex in examples:
        rel = ex.get("relpath")
        if not rel:
            continue
        rebuilt = str(root / rel)
        ex["image"] = rebuilt
        for msg in ex.get("messages", []):
            for part in msg.get("content", []):
                if isinstance(part, dict) and part.get("type") == "image":
                    part["image"] = rebuilt
    return examples
''')
    print('patched    : src/prepare_data.py')
else:
    print('patched    : already applied')

for _m in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.') or m.startswith('scripts')]:
    del sys.modules[_m]

from src.utils import load_config
from src.prepare_data import load_jsonl
cfg = load_config()
_tr = load_jsonl(cfg['data']['train_file'])
_va = load_jsonl(cfg['data']['validation_file'])
_missing = [e for e in _tr + _va if not Path(e['image']).exists()]
print(f'train      : {len(_tr)} | validation: {len(_va)} | missing files: {len(_missing)}')
assert not _missing, f'missing images, e.g. {_missing[:2]}'
print('SETUP OK')


## 2. Baseline guard

The whole point of the study is base-first. This refuses to proceed without
saved baseline results.


In [ ]:
import pandas as pd

baseline_csv = PROJECT / 'results/baseline/baseline_results.csv'
assert baseline_csv.exists(), (
    'BASELINE MISSING.\n'
    'Run 02_base_model_baseline_colab.ipynb first, then copy results/baseline/ '
    'back into the project ZIP before fine-tuning.')

_b = pd.read_csv(baseline_csv)
print(f'baseline rows : {len(_b)} (expected 534)')
print(f'baseline model: {json.loads((PROJECT/"results/baseline/run_metadata.json").read_text())["model_id"]}')
assert len(_b) == 534, f'baseline incomplete: {len(_b)} rows'
print('Baseline present and complete — fine-tuning may proceed.')


## 3. Install training dependencies

`bitsandbytes` is needed for 4-bit quantization and the 8-bit optimizer.


In [ ]:
!pip install -q 'transformers>=5.0.0' 'accelerate>=1.0.0' 'peft>=0.14.0' 'bitsandbytes>=0.44.0' 'pyyaml>=6.0' 'pandas>=2.0' 'pillow>=10.0' 'matplotlib>=3.8' 'scipy>=1.11' 'scikit-learn>=1.4'
print('installed')


In [ ]:
import transformers, peft, bitsandbytes
print('transformers:', transformers.__version__)
print('peft        :', peft.__version__)
print('bitsandbytes:', bitsandbytes.__version__)


## 4. Training configuration

Read from `config/config.yaml` — not hard-coded here, so the run is reproducible.


In [ ]:
import yaml
print(yaml.dump({'training': cfg['training'], 'lora': cfg['lora']}, sort_keys=False))


## 5. SANITY CHECK (Phase 9) — run this before the long job

8 examples, 2 steps. Verifies that images load, the processor works, labels are
masked correctly, forward and backward passes run, **LoRA actually attached**,
the loss is finite, and a checkpoint saves.

A config bug caught here costs 5 minutes instead of 3 hours.


In [ ]:
import time
from src.train import run_training

t0 = time.time()
sanity = run_training(cfg, sanity=True)
SANITY_SECONDS = time.time() - t0

print()
print('=' * 60)
print('SANITY CHECK RESULT')
print('=' * 60)
print(f"examples used    : {sanity['n_train_examples']}")
print(f"steps completed  : {sanity['global_step']}")
print(f"training loss    : {sanity['training_loss']:.4f}")
print(f"trainable params : {sanity['trainable_params']:,} ({sanity['trainable_pct']}%)")
print(f"total params     : {sanity['total_params']:,}")
print(f"adapter saved to : {sanity['adapter_dir']}")
print(f"wall time        : {SANITY_SECONDS:.0f} s")
print('=' * 60)

assert sanity['trainable_params'] > 0, 'LoRA did NOT attach — stop and report this.'
assert sanity['training_loss'] == sanity['training_loss'], 'loss is NaN'

SEC_PER_STEP = SANITY_SECONDS / max(sanity['global_step'], 1)
import math
n_steps = math.ceil(len(_tr) / (cfg['training']['batch_size'] *
                                cfg['training']['gradient_accumulation_steps'])) \
          * cfg['training']['num_epochs']
print(f'\nestimated steps for full run : {n_steps}')
print(f'measured seconds per step    : {SEC_PER_STEP:.1f} s  (includes model load — full run will be faster per step)')
print(f'ROUGH projected training time: {SEC_PER_STEP * n_steps / 60:.0f} min')
print('\nSTOP HERE. Review before approving the full run.')


> **If the sanity check fails, stop and report the exact error.** Do not change
> the model, the data, or the evaluation settings to get past it.
>
> Two likely failures:
> - `trainable_params == 0` — LoRA target module names do not match this model.
> - **CUDA OOM** — lower `model.max_pixels` in `config/config.yaml`
>   (602112 → 401408). This changes the vision-token budget, so the fine-tuned
>   evaluation **must** use the same value, and the change must be recorded in
>   `PROJECT_DECISIONS.md`.


## 5b. APPROVAL GATE — full training does not start automatically


In [ ]:
# ---------------------------------------------------------------
# Set to True ONLY after reviewing the sanity check above.
APPROVED_FOR_FULL_TRAINING = False
# ---------------------------------------------------------------

if not APPROVED_FOR_FULL_TRAINING:
    print('FULL TRAINING NOT APPROVED — stopping here by design.')
    print(f'Rough projection was {SEC_PER_STEP * n_steps / 60:.0f} min for {n_steps} steps.')
    print('Set APPROVED_FOR_FULL_TRAINING = True above, re-run this cell, then continue.')
else:
    print(f'Approved. Proceeding with ~{n_steps} steps.')


## 6. Full LoRA training

Checkpoints save every 100 steps into `results/training/checkpoints/`, so a
disconnect does not lose everything.

> **Keep this tab active.** Free Colab disconnects idle sessions, and this is
> the longest job in the project.


In [ ]:
assert APPROVED_FOR_FULL_TRAINING, (
    'Full training not approved. Review the sanity check, then set '
    'APPROVED_FOR_FULL_TRAINING = True in the cell above.')

import gc
gc.collect(); torch.cuda.empty_cache()
print('VRAM before training:', round(torch.cuda.memory_allocated()/1e9, 2), 'GB')

t0 = time.time()
summary = run_training(cfg, sanity=False)
TRAIN_SECONDS = time.time() - t0

print()
print('=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)
print(f"final training loss : {summary['training_loss']:.4f}")
print(f"steps               : {summary['global_step']}")
print(f"trainable params    : {summary['trainable_params']:,} ({summary['trainable_pct']}%)")
print(f"adapter             : {summary['adapter_dir']}")
print(f"wall time           : {TRAIN_SECONDS/60:.1f} min")
print('=' * 60)


## 7. Training curves


In [ ]:
from src.train import plot_training_curves
from IPython.display import Image, display

curve = plot_training_curves(summary['log_history'], 'results/training/training_curves.png')
if curve:
    display(Image(str(curve)))
else:
    print('no loss entries logged')

hist = [h for h in summary['log_history'] if 'loss' in h or 'eval_loss' in h]
pd.DataFrame(hist).tail(15)


## 8. Verify the adapter saved and reloads


In [ ]:
adapter_dir = Path(summary['adapter_dir'])
print('adapter dir:', adapter_dir)
total = 0
for f in sorted(adapter_dir.iterdir()):
    if f.is_file():
        total += f.stat().st_size
        print(f'  {f.name:<44}{f.stat().st_size:>12,} bytes')
print(f'  {"TOTAL":<44}{total:>12,} bytes ({total/1e6:.1f} MB)')

assert (adapter_dir / 'adapter_config.json').exists(), 'adapter_config.json missing'
print('\nadapter saved correctly')


## 9. Package and download the adapter

> **Do this before the runtime disconnects.** `/content` is wiped when the
> session ends, and the whole training run would be lost.

The adapter is small — LoRA stores only the low-rank deltas, not the 8 GB base model.


In [ ]:
import shutil

archive = shutil.make_archive('/content/finetuned_adapter', 'zip',
                              root_dir=str(adapter_dir))
print(f'wrote {archive} ({Path(archive).stat().st_size/1e6:.1f} MB)')

# also grab the training logs
log_dir = PROJECT / 'results' / 'training' / 'logs'
if log_dir.exists():
    shutil.make_archive('/content/training_logs', 'zip', root_dir=str(log_dir))
    print('wrote /content/training_logs.zip')

if IN_COLAB:
    from google.colab import files
    for f in ['/content/finetuned_adapter.zip', '/content/training_logs.zip']:
        if Path(f).exists():
            try:
                files.download(f)
            except Exception as e:
                print('download failed for', f, '->', e)
    print('If downloads are blocked, use the file browser (folder icon).')


## 10. STOP HERE

Training is done. **Do not run the evaluation in this notebook** — it belongs
in `05_evaluation`, which reloads the *frozen* evaluation subset and the same
prompts the baseline used.

Next:
1. Confirm `finetuned_adapter.zip` downloaded.
2. Unzip it locally into `results/training/final_model/`.
3. Re-zip the project and run the fine-tuned evaluation on the same 178 images.

```bash
cd ~/casting-defect-vlm
mkdir -p results/training/final_model
unzip -o ~/Downloads/finetuned_adapter.zip -d results/training/final_model/
```

> The fine-tuned evaluation **must** use the identical images, prompts, seed and
> decoding settings as the baseline, or the comparison is invalid.
> `compare_models.py` aborts if the two runs differ in even one image.
